# 09 — C-NMC source inspection

**Purpose:** Discover labeled training folds, decode cells, audit counts and patient IDs.

**Required inputs:** C-NMC `training_data/fold_0..2/{all,hem}` at cnmc_root or beneath it.

**Saved outputs:** cnmc_manifest.csv, dataset_inspection.json, sample and class-distribution figures.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [ ]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

## 2. Verify source and metadata

The official 10,661-image count is a comparison, not an assumed local result.

In [ ]:
from src.cnmc_data import inspect_cnmc
import pandas as pd
frame, report = inspect_cnmc(cfg)
display(pd.DataFrame([{"metric": key, "value": str(value)} for key, value in report.items() if key != "errors"]))
display(frame.groupby(["fold", "class_name"]).size().rename("cells").reset_index())
display(frame.groupby(["subject_id", "class_name"]).size().rename("cells").reset_index().head(20))
display(frame[["image_width", "image_height", "image_channels"]].describe())

## 3. Save labeled gallery

Eight normal and eight blast examples are selected when available.

In [ ]:
from src.cnmc_data import sample_figures
from IPython.display import Image as DisplayImage, display
sample_figures(frame, out / "figures")
display(DisplayImage(filename=str(out / "figures" / "cnmc_samples.png")))
display(DisplayImage(filename=str(out / "figures" / "class_distribution.png")))

## 4. Integrity summary

Invalid subject names, damaged images, or duplicate image content block splitting.

In [ ]:
assert report["ready"] and frame.class_id.isin([0, 1]).all()
print("PASS — source untouched; manifest saved. Official count match:", report["matches_official_counts"])

## Next

Open **10_CNMC_Patient_Split.ipynb**; never split images randomly.